## **ESTUDIO DE APOPHIS CON LOS PROBLEMAS DE 2, 3 Y N CUERPOS**

In [ ]:
import pymcel as pc
import numpy as np
import rebound as rb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.animation import FuncAnimation
from astropy.time import TimeDelta
from astropy.time import Time
from IPython.display import HTML
import plotly.graph_objects as go
from pymcel.constantes import GM_earth
from pymcel.constantes import GM_sun
from pymcel.constantes import M_sun
from pymcel.constantes import M_earth
import warnings
warnings.filterwarnings('ignore')

## Trayectoria de Apophis en un sistema de N Cuerpos

Es evidente que en el movimiento de los cuerpos en el espacio influyen todas las partículas con masa que estén a una distancia medianamente cercana, por lo que se usará primeramente esta teoría para calcular la distancia más cercana que tendrá el cometa respecto a la tierra, teniendo en cuenta todos los planetas del sistema solar y la luna. 

La fuerza gravitacional que experimenta un cuerpo debido a N cuerpos es: 
$$
\vec{F_i} 
=
m_i \ddot{\vec{r}_i}
=
\sum_{j \neq i}
G
\frac{m_i m_j}{|\vec{r}_j-\vec{r}_i|^3}
(\vec{r}_j-\vec{r}_i)
$$

Esta teoría se aplicará por medio de la creación de una simulación del sistema solar, integrando numéricamente y así encontrar las posiciones $ x(t), y(t), z(t) $ de los cuerpos de interés (que en este caso son la tierra y Apophis) que surgen de solucionar la ecuación diferencial definida.

Para calcular la distancia mínima entre estos dos objetos, luego de obtener las posiciones para cada tiempo, basta con hallar la magnitud del vector relativo entre ambos cuerpos:

$$
r(t) = \sqrt{(x_1 - x2)^2 + (y_1 - y_2)^2 + (z_1 - z_2)^2}
$$

y observar el t (que es la fecha de máximo acercamiento) donde este valor es el menor.

## Unidades Canónicas

El código a continuación utiliza las siguientes unidades canónicas para simplificar el orden de los datos a obtener:

$U_L =  1 $ Unidad Astronómica (AU)

$U_M = 1 $ Masa solar($M_{sun}$)

Tomando los valores de una unidad astronómica y una masa solar, para despejar la unidad canónica del tiempo se hace:

$
U_T  = \sqrt{\frac{U_L ^3}{G U_M}}
$

# Integración numérica

Para integrar numéricamente la ecuación de movimiento para un sistema de N cuerpos se usará el integrador IAS15 de Rebound, que es ideal para sistemas como el que queremos estudiar por su alta precisión en problemas que tienen valores de tiempo muy distintos.

La simulación que se creará será de todos los planetas del sistemas solar (hasta Urano), la luna y, evidentemente, Apophis.

Para que el integrador pueda ejectutar correctamente su función se necesita escoger un origen de coordenadas, éste será el centro de masa del sistema solar que se sabe que se mueve con velocidad constante y donde se tiene certeza que las leyes de Newton pueden aplicarse.

## Ejecución del código

Para llevar esto a cabo se necesita crear un sistema dentro de rebound incluyendo todos los cuerpos que queremos considerar en la integración junto a sus masas. Le entregamos las unidades canónicas que definimos anteriormente para que trabaje con ellas. Le pasamos la fecha en que queremos que comience a hacer la integración, la cantidad de días que queremos que dure, y qué tan seguido queremos que se tomen los datos.

Vamos a integrar el sistema durante 4 años, comenzando en 2027 y terminando en 2031 con el fin de que el momento de menor distancia quede aproximadamente en la mitad. Tomaremos un paso de tiempo de 1 hora para obtener unos datos precisos. Todos los datos que se entreguen de tiempo, masa o longitud se deben poner en los términos de unidades canónicas que ya se definieron para el integrador (segundos, masas solares y unidades astronómicas)



Vamos a definir las unidades canónicas

In [ ]:
UL = 1.496e+11  # Unidad Astronómica en metros
UM = 1.989e+30  # Masa del Sol en kg
G = 6.67430e-11  # Constante de gravitación universal en m^3 kg^-1 s^-2
UT = (UL**3 / (G * UM))**0.5  # Unidad de tiempo

In [ ]:
# Creamos la simulación del sistema solar
sim = rb.Simulation()

#Seteamos las unidades
sim.units = ('s', 'AU', 'Msun')

#Definimos el integrador
sim.integrator = 'ias15'

#Definimos la fecha inicial de la simulación desde Time
fecha_inicial = '2027-01-01 00:00:00'

#
cuerpos = ['Sun', 'Apophis','301', 'Mercury', 'Venus', '399', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']

for cuerpo in cuerpos:
    sim.add(cuerpo, date=fecha_inicial)

sim.move_to_com()  # Mover origen al centro de masa del sistema solar

Ya que tenemos el sistema creado vamos a hacer la integración numérica con un paso temporal de una hora. Vamos a crear arrays vacíos para guardar los tiempos, las posiciones y las velocidades. Luego creamor un ciclo for para que se vayan guardando las posiciones y velocidades de los cuerpos en todos los tiempos

In [ ]:
tiempo_total = 4 * 365.25 * 24 * 3600  # 4 años en segundos
dt = 1 * 3600  # Paso de tiempo de 1 hora en segundos
paso = int(tiempo_total / dt)  # Número total de pasos de la simulación

tiempos = np.zeros(paso + 1) #Almacena los tiempos de cada paso
posiciones = np.zeros((paso + 1, sim.N, 3))  # Almacena las posiciones
velocidades = np.zeros((paso + 1, sim.N, 3)) # Almacena las velocidades

for paso, t in enumerate(np.linspace(0, tiempo_total, paso + 1)):
    sim.integrate(t)

    tiempos[paso] = t # Guardar el tiempo actual en el array de tiempos
    
    for cuerpo in range(sim.N): # Iterar sobre cada cuerpo en la simulación y guardar sus posiciones y velocidades
        posiciones[paso, cuerpo] = [sim.particles[cuerpo].x, 
        
                                    sim.particles[cuerpo].y, 
                                    sim.particles[cuerpo].z]
        
        velocidades[paso, cuerpo] = [sim.particles[cuerpo].vx, 
                                     sim.particles[cuerpo].vy, 
                                     sim.particles[cuerpo].vz]

Como el fin de la integración es ver cuál es la distancia mínima que tendrá Apophis respecto a la tierra, vamos a aislar las posiciones de los dos cuerpos para restarlas y ver en qué momento esta diferencia es menor, que sería la distancia mínima entre los dos cuerpos con su respectiva fecha

In [ ]:
x_apo = posiciones[:, 1, 0]  # Posición x de Apophis
y_apo = posiciones[:, 1, 1]  # Posición y de Apophis
z_apo = posiciones[:, 1, 2]  # Posición z de Apophis

x_earth = posiciones[:, 5, 0]  # Posición x de la Tierra
y_earth = posiciones[:, 5, 1]  # Posición y de la Tierra
z_earth = posiciones[:, 5, 2]  # Posición z de la Tierra

dx = x_apo - x_earth # Diferencia en la posición x entre Apophis y la Tierra
dy = y_apo - y_earth # Diferencia en la posición y entre Apophis y la Tierra
dz = z_apo - z_earth # Diferencia en la posición z entre Apophis y la Tierra

distancias = np.sqrt(dx**2 + dy**2 + dz**2)  # Distancia entre Apophis y la Tierra en cada paso

indice_min = np.argmin(distancias)  # Índice de Distancia mínima entre Apophis y la Tierra
distancia_min = distancias[indice_min]  # Distancia mínima entre Apophis y la Tierra
tiempo_min = tiempos[indice_min]  # Tiempo en el que ocurre la distancia mínima

distancia_min_km = distancia_min * UL / 1000  # Convertir distancia mínima a kilómetros

fecha_min = Time(fecha_inicial) + TimeDelta(tiempo_min, format='sec') # Calcular la fecha de mínimo acercamiento sumando el tiempo mínimo a la fecha inicial

print(f"Distancia mínima: {distancia_min_km:.2f} km")
print(f"Fecha de mínimo acercamiento: {fecha_min.iso}")


In [ ]:

indice_min = np.argmin(distancias)  # Índice de Distancia mínima entre Apophis y la Tierra
distancia_min = distancias[indice_min]  # Distancia mínima entre Apophis y la Tierra
tiempo_min = tiempos[indice_min]  # Tiempo en el que ocurre la distancia mínima

distancia_min_km = distancia_min * UL / 1000  # Convertir distancia mínima a kilómetros

fecha_min = Time(fecha_inicial) + TimeDelta(tiempo_min, format='sec') # Calcular la fecha de mínimo acercamiento sumando el tiempo mínimo a la fecha inicial

print(f"Distancia mínima: {distancia_min_km:.2f} km")
print(f"Fecha de mínimo acercamiento: {fecha_min.iso}")


Comparando este dato obtenido con el dato oficial en JPL Small-Body Database (38008 km el 13 de abril a las 21:46) vemos que el que resultó de aplicar la teoría de N cuerpos es bastante acertado con el real (aunque quizá ayudó el gran intervalo temporal que se tomó), teniendo en cuenta que no se agregaron todos los cuerpos con masa relativamente grande del sistema solar (como las lunas de júpiter o de marte, por ejemplo)

## Análisis

Analizando los datos podemos concluir que, aunque no se agregaron todos los cuerpos del sistema solar sino unos cuantos, el resultado obtenido aplicando la teoría de N cuerpos al cometa Apophis es bastante acertado, permitiendo afirmar que los planetas del sistema solar, el sol y la luna rigen casi por completo el movimiento de los cuerpos de masa pequeña en el sistema solar

## Animación del paso de Apophis

Creemos un código para ilustrar el paso cercano de Apophis por la tierra. Para esto, se crea la animación posicionando al sol en el centro (que en la práctica es restar las coordenadas de todos los cuerpos a las del sol) y tomando solamente los planetas hasta Marte para no hacer la animación muy grande.

In [ ]:
# 1. Transformar posiciones a coordenadas heliocéntricas (relativas al Sol)
# Restamos la posición del Sol (índice 0) a todos los cuerpos
pos_helio = posiciones - posiciones[:, 0:1, :]

# Configuración de la figura en negro para emular el espacio
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_facecolor('black')
fig.patch.set_facecolor('black')

# Cuerpos a graficar: {índice: (nombre, color, tamaño_marcador)}
cuerpos_helio = {
    1: ('Apophis', '#ff3333', 5),
    3: ('Mercurio', '#888888', 4),
    4: ('Venus', '#ffcc00', 7),
    5: ('Tierra', '#3399ff', 8),
    6: ('Marte', '#ff6600', 6)
}

# Dibujar el Sol fijo en el origen
ax.plot(0, 0, 'o', color='#ffff33', markersize=14, label='Sol', zorder=5)

lineas = {}
puntos = {}

# Dibujar órbitas completas (estáticas y semi-transparentes) y configurar animaciones
for idx, (nombre, color, size) in cuerpos_helio.items():
    # Órbita completa de fondo
    ax.plot(pos_helio[:, idx, 0], pos_helio[:, idx, 1], color=color, alpha=0.15, linestyle=':', linewidth=1)
    
    # Línea que dibuja la estela animada del cuerpo
    lineas[idx], = ax.plot([], [], color=color, alpha=0.5, linewidth=1.5)
    
    # Punto que representa el planeta/asteroide actual
    puntos[idx], = ax.plot([], [], 'o', color=color, markersize=size, label=nombre)

# Configuración estética de los límites y etiquetas
ax.set_xlim([-2.0, 2.0]) # Limita a 2 UA para enfocar el sistema interno
ax.set_ylim([-2.0, 2.0])
ax.set_aspect('equal')
ax.grid(True, color='gray', alpha=0.2, linestyle='--')
ax.set_title("Órbitas en el Sistema Solar Interno (Heliocéntrico, 2026-2032)", color='white', fontsize=14, pad=15)
ax.set_xlabel("X (UA)", color='white')
ax.set_ylabel("Y (UA)", color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='black', edgecolor='gray', labelcolor='white', loc='upper right')

# Texto para mostrar la fecha actual
txt_fecha = ax.text(0.02, 0.95, '', color='white', transform=ax.transAxes, fontsize=12, fontweight='bold')

def init_helio(): # Función de inicialización para la animación
    for idx in lineas:
        lineas[idx].set_data([], [])
        puntos[idx].set_data([], [])
    txt_fecha.set_text('')
    return list(lineas.values()) + list(puntos.values()) + [txt_fecha]

# Muestreamos un cuadro cada 100 horas (100 pasos de 1 hora) para no sobrecargar la memoria
skip_helio = 150
frames_helio = range(0, len(tiempos), skip_helio)

def update_helio(frame): # Función de actualización para cada cuadro de la animación
    current_date = Time(fecha_inicial) + TimeDelta(tiempos[frame], format='sec')
    txt_fecha.set_text(f"Fecha: {current_date.iso.split()[0]}")
    
    for idx in lineas:
        # Dibujamos la estela de los últimos 20 días para mayor fluidez
        inicio_estela = max(0, frame - 500)
        lineas[idx].set_data(pos_helio[inicio_estela:frame, idx, 0], pos_helio[inicio_estela:frame, idx, 1])
        puntos[idx].set_data([pos_helio[frame, idx, 0]], [pos_helio[frame, idx, 1]])
        
    return list(lineas.values()) + list(puntos.values()) + [txt_fecha]

anim_helio = FuncAnimation(fig, update_helio, frames=frames_helio, init_func=init_helio, blit=True, interval=30)
plt.close(fig) # Cierra la visualización estática para mostrar solo el HTML interactivo

HTML(anim_helio.to_jshtml())

Se observa efectivamente el acercamiento que tiene Apophis en los meses iniciales de 2026 para luego 'adelantar' a la tierra. Nótese cómo Apophis cambia drásticamente su órbita a una más ancha luego de su encuentro con la tierra. Complementemos esta consideración con un gráfico que muestre la distancia entre la tierra y Apophis.

Usando la variable 'distancias' que creamos antes, que es la que calcula las diferencias de distancias entre Apophis y la tierra, convertida en kilómetros podemos graficar las distancias que va teniendo Apophis respecto a la tierra a lo largo del tiempo de integración. Para tener la fecha en el eje X, vamos a convertir los intervalos de tiempo que están en segundos en fechas específicas partiendo de la fecha inicial que definimos. Además se hará una distinción del punto de menor acercamiento entre Apophis y la tierra que fue el que ya hallamos 

In [ ]:
# 1. Convertir el array de tiempos (segundos desde la fecha inicial) a objetos datetime
fechas_astropy = Time(fecha_inicial) + TimeDelta(tiempos, format='sec')
fechas_datetime = fechas_astropy.datetime

# 2. Convertir las distancias de Unidades Astronómicas (AU) a kilómetros
distancias_km = distancias * UL / 1000.0

# 3. Crear el gráfico
plt.figure(figsize=(10, 6), dpi=100)
plt.plot(fechas_datetime, distancias_km, color='#1f77b4', linewidth=2, label='Distancia Apophis - Tierra')

# 4. Señalar el punto de mínimo acercamiento en el gráfico
fecha_min_datetime = fecha_min.datetime
plt.scatter(fecha_min_datetime, distancia_min_km, color='red', s=80, zorder=5, 
            label=f'Mínimo acercamiento\n({fecha_min.iso.split(".")[0]} UT: {distancia_min_km:.2f} km)')

# 5. Formatear y embellecer el gráfico
plt.title('Variación de la Distancia Apophis - Tierra (2027 - 2031)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Distancia (km)', fontsize=12)

# Formatear el eje X para mostrar las fechas de manera legible
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.gcf().autofmt_xdate() # Rotar las fechas para que no se traslapen

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=10, loc='upper right')
plt.tight_layout()

# Mostrar el gráfico
plt.show()


## Análisis

El comportamiento de la distancia entre Apophis y la tierra se debe a que el periodo orbital de Apophis es de unos 323 días, y el de la tierra es de 365, entonces cuando ambos se encuentran en el mismo sector de sus respectivas órbitas, la distancia disminuye y se genera un valle (mínimo) en la gráfica. Por el contrario, cuando se encuentran en extremos opuestos respecto al Sol (oposición), la distancia aumenta, formando una cresta (máximo). Y como las órbitas no tienen el mismo periodo, surgen los valles y crestas más altos (o bajos) en algunos momentos.

## Momentum Angular

En un sistema de N cuerpos respecto a un sistema de referencia inercial, el momentum angular se define como 

$$\vec{L} = \sum_{i=1}^{N} m_i\vec{r_i} \times \dot{\vec{r_i}}$$

Suponiendo que el sistema (en este caso el sistema solar) está aislado del universo, a partir de la ecuación del movimiento para los N cuerpos, se puede encontrar una cuadratura para el sistema, que es precisamente el momentum angular (y el que sea una cuadratura quiere decir que éste es constante). 

## Momentum angular específico relativo 
En el problema de dos cuerpos, si se considera el movimiento de un cuerpo de masa $m_i$ respecto a otro de masa $m_j$, se puede definir el vector de posición relativo $\vec{r} = \vec{r_i} - \vec{r_j}$ y la velocidad relativa $\dot{\vec{r}} = \dot{\vec{r_i}} - \dot{\vec{r_j}}$. A partir de estos, el momentum angular específico relativo se define como:

$$ \vec{h} = \vec{r} \times \dot{\vec{r}} $$

A diferencia del momentum angular total $\vec{L}$, este es un momentum angular por unidad de masa y describe la geometría de la órbita relativa entre dos cuerpos. Suponiendo una vez más que sólo existen dos cuerpos en el universo, $\vec{h}$ también es una cuadratura del movimiento. Que el vector de momentum angular específico relativo sea constante, quiere decir que el plano que definen los dos cuerpos que estamos considerando es siempre constante, y es lo que notaremos en el caso de Apophis y el sol.

Comprobemos con los datos ya obtenidos de las posiciones por medio de la teoría de los N cuerpos, que el momentum angular de este sistema se conserva. Para eso debemos obtener las masas de los cuerpos del sistema que tenemos (los planetas, el sol, la luna y Apophis), calcular el momentum angular de cada una y luego sumarlas para obtener el total. Para saber su valor calculamos la magnitud del vector.

Para el caso del MARE de Apophis y el sol, calculamos el vector relativo entre ambos calculando la resta de sus coordenadas y luego aplicamos la ecuación. Para obtener el resutado en kilómetros calculamos la norma del vector y multiplicamos por la unidad canónica.


In [ ]:
# 1. Obtener las masas de los cuerpos desde la simulación (en masas solares)
masas = np.array([sim.particles[i].m for i in range(sim.N)])

# 2. Calcular el Momentum Angular Total del sistema (L_total)
L_individual = masas[np.newaxis, :, np.newaxis] * np.cross(posiciones, velocidades)# np.cross opera sobre el último eje (las componentes x, y, z) de posiciones y velocidades
L_total = np.sum(L_individual, axis=1) # Sumamos sobre todos los cuerpos
L_total_mag = np.linalg.norm(L_total, axis=1) # Magnitud de L_total en Msun * AU^2 / s}

# 3. Calcular el Momentum Angular Específico de Apophis respecto al Sol (h_helio)

r_apo_sun = posiciones[:, 1, :] - posiciones[:, 0, :] # Apophis: índice 1, Sol: índice 0
v_apo_sun = velocidades[:, 1, :] - velocidades[:, 0, :]
h_helio_vec = np.cross(r_apo_sun, v_apo_sun)
h_helio_mag_au = np.linalg.norm(h_helio_vec, axis=1) # en AU^2 / s
h_helio_mag_km = h_helio_mag_au * (UL / 1000)**2 # en km^2 / s

# Convertir tiempos en segundos a fechas para graficar
fechas_time = Time(fecha_inicial) + TimeDelta(tiempos, format='sec')
fechas_dt = fechas_time.datetime

Vamos a graficar ambos momentos angulares para observar su comportamiento durante el paso de Apophis. Para esto creamos un plot de dos gráficas en una columna

In [ ]:
# Graficación del Momentum Angular
fig, axs = plt.subplots(2, 1, figsize=(12, 15), sharex=False)
fig.suptitle('Evolución del Momentum Angular en la Simulación de Apophis', fontsize=16, fontweight='bold', color='#2c3e50')

# Colores elegantes para cada gráfica
color_total = '#2980b9'  # Azul
color_helio = '#27ae60'  # Verde
color_vline = '#8e44ad'  # Violeta para la fecha del encuentro

#Momentum Angular Total del Sistema
axs[0].plot(fechas_dt, L_total_mag, color=color_total, linewidth=2, label=r'$|\vec{L}_{total}|$')
axs[0].set_title('1. Momentum Angular Total del Sistema (Verificación de Conservación)', fontsize=12, fontweight='bold')
axs[0].set_ylabel(r'$L_{total}$ ($M_{\odot} \cdot \mathrm{AU}^2/\mathrm{s}$)', fontsize=10)
axs[0].grid(True, linestyle='--', alpha=0.5)
axs[0].ticklabel_format(axis='y', useOffset=False, style='scientific')
axs[0].legend(loc='upper right')

# Momentum Angular Heliocéntrico de Apophis
axs[1].plot(fechas_dt, h_helio_mag_km, color=color_helio, linewidth=2, label=r'$|\vec{h}_{helio}|$ (Apophis-Sol)')
axs[1].set_title('2. Momentum Angular Específico de Apophis respecto al Sol (Cambio Orbital)', fontsize=12, fontweight='bold')
axs[1].set_ylabel(r'$h_{helio}$ ($\mathrm{km}^2/\mathrm{s}$)', fontsize=10)
axs[1].grid(True, linestyle='--', alpha=0.5)
axs[1].axvline(x=fecha_min.datetime, color=color_vline, linestyle=':', linewidth=2, 
               label=f'Máximo acercamiento ({fecha_min.iso[:10]} {fecha_min.iso[11:16]})')
axs[1].legend(loc='best')


# Formatear los ejes
for ax in axs:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Fecha', fontsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Análisis

Vemos que, efectivamente el momentum angular del sistema solar (la parte que consideramos) se conserva aún con el cambio de órbita que Apophis sufrió debido al empujón de la tierra, mostrando su cualidad de cuadratura de movimiento. 

Para el plano que determinan las trayectorias de Apophis y el sol vemos que se mantiene constante hasta que hay un salto abrupto que cambia el valor de este plano para luego continuar constante. Evidentemente el momento en el que sucede este salto es en el acercamiento de Apophis con la tierra, el cual al recibir un empujón por parte de la tierra y cambiar su órbita (como se vio en la animación anterior) también cambia la configuración que tenía respecto al sol, provocando el salto y permaneciendo constante luego de esto.


## **Energías potencial, cinética y total**
En un sistema de N cuerpos, la energía cinética del sistema se calcula como la suma de cada una de las energías potenciales de los cuerpos, a saber:

**Energía Cinética ($K$):** $$K = \frac{1}{2} \sum_{i} m_i \dot{\vec{r_i}}^2$$

Y la energía potencial gravitacional (que se debe a la atracción gravitacional, valga la redundancia, entre dos cuerpos) se calcula como la energía potencial gravitacional entre todos los cuerpos entre sí por pares:

 **Energía Potencial Gravitacional ($U$):** $$U = -\sum_{i < j} \frac{m_i \mu_j}{r_{ij}}$$
 con $\mu_j = G m_j$ y $r_{ij} = ||\mathbf{r}_i - \mathbf{r}_j\|$
 Por último, La energía total $E$ se calcula como:

$$E = K + U$$

Al estar suponiendo un sistema aislado que, además sólo está bajo la influencia de fuerzas conservativas (gravitatorias), a las fuerzas pueden atribuirse un trabajo, que, entre dos puntos, es igual al cambio en la energía total del sistema por el teorema del trabajo-energía. Por otro lado, la energía mecánica total de un sistema de N cuerpos es una constante de movimiento, es decir, que es constante. En las siguientes celdas calcularemos la energía cinética, potencial y total del sistema solar junto con Apophis.



Para calcular las energias crearemos arrays vacíos donde guardaremos eventualmente las energías; escalamos las velocidades de modo que nos queden en unidades canónicas adecuadamente y poder usar G = 1. Creamos un ciclo for para calcular las energías cinéticas de cada paso temporal. Para calcular las energías potenciales creamos dos ciclos de modo que uno siempre estuviera en un índice mayor al otro para calcular las energías correctamente; y la energía mecánica es simplemente la suma de estas dos listas. Por último se crearon 3 gráficos para representar la evolución temporal de las energías.

In [ ]:
num_pasos = posiciones.shape[0]
velocidades_canonicas = velocidades * UT  # Convertir de AU/s a AU/UT (unidades canónicas)

energia_cinetica = np.zeros(num_pasos)
energia_potencial = np.zeros(num_pasos)

for paso in range(num_pasos):
    pos = posiciones[paso] # Guardamos las posiciones de todos los cuerpos en el paso actual
    vel = velocidades_canonicas[paso]

    # Energía Cinética: K = sum(1/2 * m_i * v_i^2)
    v2 = np.sum(vel**2, axis=1)
    energia_cinetica[paso] = 0.5 * np.sum(masas * v2)

    # Energía Potencial: U = -sum_{i<j} G * m_i * m_j / r_ij
    u = 0.0
    for i in range(len(masas)):
        for j in range(i + 1, len(masas)):
            r = np.sqrt(np.sum((pos[i] - pos[j])**2))
            if r > 0:
                u -= masas[i] * masas[j] / r
    energia_potencial[paso] = u

# Energía Mecánica Total: E = K + U
energia_mecanica = energia_cinetica + energia_potencial

fechas_astropy = Time(fecha_inicial) + TimeDelta(tiempos[:num_pasos], format='sec')
fechas_dt = fechas_astropy.datetime

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 12), dpi=100, sharex=True)

# --- Energía Cinética ---
ax1.plot(fechas_dt, energia_cinetica, color='#2ca02c', linewidth=1.5)
ax1.set_ylabel('K (canónica)', fontsize=11)
ax1.set_title('Energía Cinética', fontsize=13, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.4)

# --- Energía Potencial ---
ax2.plot(fechas_dt, energia_potencial, color='#d62728', linewidth=1.5)
ax2.set_ylabel('U (canónica)', fontsize=11)
ax2.set_title('Energía Potencial', fontsize=13, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.4)

# --- Energía Mecánica Total ---
ax3.plot(fechas_dt, energia_mecanica, color='#1f77b4', linewidth=1.5)
ax3.set_ylabel('E (canónica)', fontsize=11)
ax3.set_title('Energía Mecánica Total (E = K + U)', fontsize=13, fontweight='bold')
ax3.set_xlabel('Fecha', fontsize=11)
ax3.grid(True, linestyle='--', alpha=0.4)

# Formatear el eje X compartido
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
fig.autofmt_xdate()

fig.suptitle('Evolución de las Energías del Sistema (Unidades Canónicas)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Análisis

La gráfica muestra una antisimetría entre las energías potencial y gravitacional lo cual tiene sentido con la realidad: a medida que un cuerpo va aumentando su velocidad (ya sea por estar en el perihelio o por experimentar algún empujón gravitacional), su energía potencial se va consumiendo y viceversa.

El gráfico de la energía mecánica total muestra que su valor se mantiene esencialmente constante a lo largo de toda la simulación, con fluctuaciones del orden de $10^{-9}$. Estas pequeñísimas variaciones no se deben a una falla en la integración numérica, sino a los límites de precisión de la aritmética de punto flotante al calcular y restar dos cantidades de magnitud similar ($K$ y $U$). Esto confirma que el integrador IAS15 conserva correctamente la energía del sistema, validando la eficacia de la simulación del problema de N cuerpos.



## **El virial y la Identidad de Lagrange Jacobi**

**Teorema del Virial**

Para un sistema gravitacional ligado gravitacionalmente (como el sistema solar), el Teorema del Virial establece una relación entre los promedios temporales de la energía cinética y la energía potencial:

$$\langle 2K \rangle = -\langle U \rangle$$

Esto implica que, en promedio, la magnitud de la energía potencial es el doble de la energía cinética, y que la energía total del sistema satisface $\langle E \rangle = \langle K \rangle + \langle U \rangle = -\langle K \rangle = \frac{1}{2}\langle U \rangle < 0$. Si esta condición se cumple, el sistema se encuentra gravitacionalmente ligado y es estable en el tiempo. Si por el contrario $2K > |U|$, el sistema tiene energía suficiente para disgregarse.

**Identidad de Lagrange-Jacobi**

La Identidad de Lagrange-Jacobi permite estudiar la estabilidad dinámica de un sistema de N cuerpos a través del momento de inercia escalar del sistema respecto a su centro de masa:

$$I = \sum_{i} m_i , |\vec{r}_i|^2$$

La identidad relaciona la segunda derivada temporal de $I$ con las energías cinética y potencial del sistema:

$$\frac{1}{2}\ddot{I} = 2K + U$$


Esta relación es una herramienta para determinar la estabilidad del sistema: si $\ddot{I} < 0$ de forma sostenida, el sistema tiende a contraerse (colapso); si $\ddot{I} > 0$, el sistema se expande (disgregación). En un sistema estable y virializado, $\ddot{I}$ oscila alrededor de cero, lo que indica que el sistema ni colapsa ni se dispersa, manteniéndose confinado en una región acotada del espacio.

Veamos si el sistema que estamos estudiando cumple con la identidad de Lagrange - Jacobi, para eso grafiquemos $\ddot{I}$ en el tiempo y si oscila al rededor de 0, podremos afirmar que el sistema es estable a largo plazo.


In [ ]:
I_dp = 2 * energia_cinetica + energia_potencial
plt.figure(figsize=(10, 5), dpi=100)
plt.plot(fechas_dt, I_dp, color='#ff7f0e', linewidth=1.5)
plt.ylabel('$\ddot{I}$ (canónica)', fontsize=11)
plt.title('Evolución de $\ddot{I}$', fontsize=13, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Análisis

La gráfica de $\frac{1}{2}\ddot{I} = 2K + U$ muestra que esta cantidad se mantiene negativa a lo largo de toda la simulación, con valores del orden de $10^{-6}$. Dado que $K$ y $U$ individualmente son del orden de $10^{-4}$, el hecho de que su combinación $2K + U$ sea unas 100 veces más pequeña indica que el sistema se encuentra muy cerca del equilibrio virial, donde $\langle 2K \rangle \approx -\langle U \rangle$.

Que $\ddot{I}$ sea siempre negativo significa que la energía potencial domina sobre el doble de la cinética ($2K < |U|$), confirmando que el sistema solar está fuertemente ligado gravitacionalmente.

Comparando con las gráficas anteriores se nota un comportamiento bastante similar entre las gráficas. Nótese que, aunque en la gráfica de la anomalía verdadera respecto a la tierra usando 'elements' parece haber un comportamiento contrario al obtenido en la gráfica encontrada manualmente, en realidad lo que hay es un ajuste, ya que en la anomalía verdadera inicial el día del acercamiento hay un salto de más de 350 grados (luego no se logra observar hasta dónde salta la gráfica), lo que sería igual a la caída a 100 grados en la gráfica obtenida con horizons, mostrando así el mismo dato solo que con diferente nombre.

## **Órbita de Apophis según la teoría del problema de 2 cuerpos**

Vamos a hacer la consideración de que en el universo existen solamente Apophis y el sol, y que gracias a esto podemos aplicar la teoría del problema de los dos cuerpos para predecir la órbita de Apophis durante el periodo de interés de nuestro proyecto. 

El vector relativo en el problema de dos cuerpos describe una cónica de ecuación
$$
r = \frac{h^2 / \mu}{1+e\cos(f)}
$$

con $h^2/\mu \equiv P$ llamado el Semilatus Rectum de la órbita, $e$ la excentricidad de la órbita y $f$ la anomalía verdadera.

El vector excentricidad se define como $$ \vec{e} = \frac{\dot{\vec{r}}\times \vec{h}}{\mu} - \frac{\vec{r}}{r}$$ y su dirección siempre es en la del periapsis (el punto de la elipse más cercano al foco)


Vamos a crear la órbita de Apophis usando esta teoría, para esto vamos a definir el vector relativo entre el sol y Apophis igual que con su velocidad relativa. Para este caso tomaremos el primer dato de las posiciones y velocidades para calcular los elementos orbitales necesarios. Vamos a calcular el momento angular específico de la misma manera que fue calculado anteriormente para hallar el semilatus rectum. 

Luego, se normalizan los vectores perpendiculares $\hat{e}, \hat{h}, \hat{q}$ para formar una base en que sea posible proyectar las posiciones de los cuerpos en el plano orbital. Siguiente a esto, se calcula la anomalía verdadera para cada posición; aquí hay que tener cuidado ya que en algunos momentos esta anomalía resultará en un cuadrante incorrecto, por lo que, si la proyección de la velocidad sobre el vector posición es menor que 0, hay que restarle $2\pi$ al valor dado. Por último definimos una elipse teórica a partir de la ecuación de la cónica que describe el vector relativo y se pasa a coordenadas cartesianas del plano orbital. Lo mismo se hace con la trayectoria obtenida a partir de las posiciones y velocidades: se proyecta en el plano orbital. Luego graficamos y observamos las discrepancias de ambas elipses.


In [ ]:

r_vec = posiciones[:, 1, :] - posiciones[:, 0, :]   # Posición relativa Apophis-Sol
v_vec = velocidades[:, 1, :] - velocidades[:, 0, :]  # Velocidad relativa Apophis-Sol
r_mag = np.sqrt(np.sum(r_vec**2, axis=1))            # Distancia Apophis-Sol

# 2. Parámetro gravitacional en unidades de simulación
mu = sim.G * masas[0]   # μ = G · M_sol

# 3. Elementos orbitales a partir de las condiciones iniciales (paso 0)
r0 = r_vec[0]
v0 = v_vec[0]
r0_mag = np.sqrt(np.sum(r0**2))

# Momento angular específico: h = r × v
h0 = np.cross(r0, v0)
h0_mag = np.sqrt(np.sum(h0**2))
# Semi-latus rectum: p = h² / μ
p = h0_mag**2 / mu

# Vector excentricidad: e = (v × h)/μ - r/r_mag
e_vec = np.cross(v0, h0) / mu - r0 / r0_mag
e = np.sqrt(np.sum(e_vec**2))

# Semieje mayor: a = p / (1 - e²)
a = p / (1 - e**2)

print(f"Excentricidad (e):       {e:.6f}")
print(f"Semi-latus rectum (p):   {p:.6f} AU")
print(f"Semieje mayor (a):       {a:.6f} AU")
print(f"|h|:                     {h0_mag:.6e}")

# Sistema de referencia en el plano orbital
#    ê → dirección del perihelio, q̂ → perpendicular en el plano, ĥ → normal
e_hat = e_vec / e
h_hat = h0 / h0_mag
q_hat = np.cross(h_hat, e_hat)

# Anomalía verdadera (f) en cada paso a partir de la posición real
cos_f = np.sum(r_vec * e_hat, axis=1) / r_mag
cos_f = np.clip(cos_f, -1, 1)
rdotv = np.sum(r_vec * v_vec, axis=1)
f_real = np.where(rdotv >= 0, np.arccos(cos_f), 2*np.pi - np.arccos(cos_f))

# Órbita kepleriana teórica: r(f) = p / (1 + e·cos(f))
r_kepler = p / (1 + e * np.cos(f_real))

# 7. Elipse teórica completa 
f_elipse = np.linspace(0, 2*np.pi, 1000)
r_elipse = p / (1 + e * np.cos(f_elipse))
x_elipse = r_elipse * np.cos(f_elipse)
y_elipse = r_elipse * np.sin(f_elipse)

# 8. Trayectoria real proyectada al plano orbital
x_real = np.sum(r_vec * e_hat, axis=1)
y_real = np.sum(r_vec * q_hat, axis=1)

# ===================== GRÁFICOS =====================

fig, ax = plt.subplots(1, 1, figsize=(16, 7), dpi=100)

# --- Gráfico 1: Órbita en el plano orbital ---
ax.plot(x_elipse, y_elipse, 'k--', linewidth=1.5, label='Órbita Kepleriana (teórica)', zorder=2)
ax.plot(x_real, y_real, color="#C28B1FE0", linewidth=1.2, label='Trayectoria N-cuerpos (simulación)', zorder=3)
ax.scatter(0, 0, color='gold', s=150, edgecolors='orange', linewidths=1.5, label='Sol', zorder=5)
ax.set_xlabel('x (AU) — dirección del perihelio', fontsize=11)
ax.set_ylabel('y (AU)', fontsize=11)
ax.set_title('Órbita de Apophis en el Plano Orbital', fontsize=13, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()


La comparación entre la trayectoria kepleriana teórica y la simulada con N‑cuerpos muestra claramente el efecto de las perturbaciones planetarias sobre la órbita de Apophis. Si bien la órbita real sigue aproximadamente la elipse de dos cuerpos, las diferencias evidencian la necesidad de incluir todos los cuerpos del sistema solar para predicciones precisas. Por otro lado, los elementos orbitales dieron muy similares a los que se encuentran en la página Small-Body Database de Nasa.

Comparemos esta órbita con la verdadera que tendrá Apophis, usemos la función de Pymcel 'Consulta_horizons' para obtener las posiciones de Apophis y el sol durante el mismo intervalo de tiempo que estudiamos, pasemos los valores a unidades astronómicas y grafiquémoslas.

In [ ]:
_, _, pos_apo_full = pc.consulta_horizons(id='Apophis', location='@SSB', epochs=dict(start='2027-01-01', stop='2031-06-06', step='1d')) #Datos de Apophis
_, _, pos_sol_full = pc.consulta_horizons(id='Sun', location='@SSB', epochs=dict(start='2027-01-01', stop='2031-06-06', step='1d')) # Datos del sol

# Calculamos el vector relativo y pasamos a km
r_rel_full = np.array(pos_apo_full[['x', 'y', 'z']] - pos_sol_full[['x', 'y', 'z']]) / 1000

# Separamos en componentes
x_full = r_rel_full[:, 0]
y_full = r_rel_full[:, 1]
z_full = r_rel_full[:, 2]

x_full_au = x_full / UL * 1000
y_full_au = y_full / UL * 1000
z_full_au = z_full / UL * 1000

plt.figure(figsize=(8, 8))
plt.plot(x_full_au, y_full_au, 'r-', linewidth=1.5, label='Órbita de Apophis (2027–2031)')
plt.scatter(0, 0, color='gold', s=200, edgecolor='orange', linewidth=2, label='Sol', zorder=5)

# Decoración
plt.xlabel('x (AU)')
plt.ylabel('y (AU)')
plt.title('Órbita de Apophis respecto al Sol')
plt.axis('equal')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## Análisis
Se observa que la órbita de Apophis calculada mediante la simulación N‑cuerpos concuerda con las efemérides reales del sistema solar proporcionadas por el sistema Horizons, lo que permite afirmar que la simulación utilizada es acertada para estudiar la evolución orbital de Apophis en el intervalo de tiempo analizado y puede extrapolarse a diferentes eventos.

## **Hodógrafo**

El vector de velocidad relativo en la órbita de Apophis respecto al sol describe un círculo que tiene centro en $(0, \mu e/h)$ y radio $mu/h$. A este círculo se le conoce como hodógrafo. La ecuación paramétrica para la velocidad es:

$$ \dot{x} = - \frac{\mu} {h} \sin(f)$$

   $$ \dot{y} = \frac{\mu}{h} (e + \cos(f))$$

   $$ \dot{z} = 0 $$


Grafiquemos el hodógrafo de la órbita de Apophis usando los elementos orbitales que ya hallamos. Como estamos usando unidades canónicas, $\mu$ = 1 


In [ ]:
# Hodógrafo
f = np.linspace(0, 2*np.pi, 35065) # 35065 pasos de 1 hora en 4 años

# Velocidades
vx = (-mu * np.sin(f)) / h0_mag
vy = (mu * (e+ np.cos(f))) / h0_mag


plt.plot(vx, vy, '.', markersize=1)
plt.plot(0, 0, 'o', color='gold', markersize=10, label='Sol')
plt.axis('equal')
plt.grid()
plt.xlabel('v_x (AU/UT)')
plt.ylabel('v_y (AU/UT)')
plt.title('Hodógrafo completo de Apophis')
plt.show()

## Análisis
Se sabe que el radio del círculo es $\mu/h$, en este caso el radio es $2.5 \times 10^{-7}  UA/UT$, multiplicando por una unidad astronómica sobre una unidad de tiempo, tenemos que el radio es aproximadamente $37 km/s$ es decir, que la velocidad de Apophis es de casi $40 km/s$

## **Ecuación de Kepler** 

La ecuación de Kepler está definida como:

$$
M = E - e\sin E
$$ 
con $M$ =$\sqrt{\frac{\mu}{a^3}}(t - t_p)$ llamada la anomalía media y $E$ llamada la anomalía excéntrica

Como evidentemente esta ecuación no tiene solución, se debe solucionar usando algún método de integración numérica, la cual usa diferentes métodos para obtener E. Cuando se sabe el valor de E, de la ecuación
$$
\tan\left(\frac{f}{2}\right)
=
\sqrt{\frac{1+e}{1-e}}
\tan\left(\frac{E}{2}\right)

$$

podemos despejar $f$ y así conocer la anomalía verdadera a partir de la excéntrica.

Veamos con esta estrategia la órbita de Apophis respecto al sol y comparémosla con las obtenidas hasta ahora.

Primero vamos a elegir un método de integración para la ecuación de Kepler, se escogerá el método Newton-Rapson ya que tiene una alta eficacia y rapidez. El paquete Pymcel tiene incorporado una función que resuelve la ecuación de Kepler proporcionándole M y $e$, por lo que se usará esa función. Para hallar el paso de Apophis por el perihelio vamos a restar las posiciones de Apophis y el sol para ver cuál es el dato menor y ese será en perihelio de la órbita.

In [ ]:
r_mag = np.sqrt(np.sum(r_vec**2, axis=1))            # Distancia Apophis-Sol

indice_min = np.argmin(r_mag)  # Índice de Distancia mínima entre Apophis y la Tierra
distancia_min = r_mag[indice_min]  # Distancia mínima entre Apophis y la Tierra
tiempo_min = tiempos[indice_min]  # Tiempo en el que ocurre la distancia mínima

distancia_min_km = distancia_min * UL / 1000  # Convertir distancia mínima a kilómetros

fecha_min = Time(fecha_inicial) + TimeDelta(tiempo_min, format='sec') # Calcular la fecha de mínimo acercamiento sumando el tiempo mínimo a la fecha inicial

print(f"Distancia mínima: {distancia_min_km:.2f} km")
print(f"Fecha de mínimo acercamiento: {fecha_min.iso}")

La fecha en que Apophis está en el perihelio es el 11 de octubre de 2027 (el $t_p$), como tenemos como referencia el día del paso cerca a la tierra de Apophis como $t$, la diferencia en segundos es la que le pondremos a M. Ya con E conocida, basta reemplazarla en la ecuación de $f

In [ ]:
n = np.sqrt(mu / a**3)  # Anomalía media (n = sqrt(μ/a³))
t_p = Time('2027-10-11 00:00:00')    # Paso por el perihelio
t   = Time('2029-04-13 21:46:00')    # Fecha del máximo acercamiento

# La resta de dos objetos Time da un TimeDelta
delta_t = (t - t_p).sec  # Diferencia en segundos

# Anomalía media
M = n * delta_t

E = pc.kepler_newton(M, e)[0]  # Resolver la ecuación de Kepler para obtener la anomalía excéntrica

f = 2*np.arctan(np.sqrt((1+e)/(1-e))*np.tan(E/2))
print(f"Anomalía verdadera (f) en el máximo acercamiento de Apophis a la tierra: {np.degrees(f):.2f}°")


Vamos a comparar las tres anomalías en un gráfico entre los días que se usaron para resolver la ecuación de Kepler. Se ponen los días en el eje X y para esto se divide el array temporal inicial que está en segundos, entre 86400 que son los segundos que tiene un día para convertir el tiempo en días. Las anomalías se calculan como se hicieron en la celda anterior.

In [ ]:
# ===================== COMPARACIÓN DE LAS TRES ANOMALÍAS =====================

# Período orbital de Apophis
T = 2 * np.pi / n  # en segundos

# Array de tiempos desde el perihelio (0 a T, un período completo)
t_array = np.linspace(0, T, 1000)
t_dias = t_array / 86400  # en días para el eje X

# Anomalía Media: M = n·t (crece linealmente)
M_array = n * t_array

# Anomalía Excéntrica: resolver Kepler M = E - e·sin(E)
E_array = np.array([pc.kepler_newton(Mi, e)[0] for Mi in M_array])

# Anomalía Verdadera
f_array = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E_array / 2),
                          np.sqrt(1 - e) * np.cos(E_array / 2))
f_array = np.where(f_array < 0, f_array + 2*np.pi, f_array)  # Llevar a [0, 2π)

# --- Valores en la fecha del máximo acercamiento
delta_t_enc = (t - t_p).sec
# Posición dentro de un período
delta_t_mod = delta_t_enc % T
t_enc_dias = delta_t_mod / 86400

# ===================== GRÁFICO =====================

plt.figure(figsize=(10, 6), dpi=100)

plt.plot(t_dias, np.degrees(M_array), color='#1f77b4', linewidth=2, label='Anomalía Media (M)')
plt.plot(t_dias, np.degrees(E_array), color='#2ca02c', linewidth=2, label='Anomalía Excéntrica (E)')
plt.plot(t_dias, np.degrees(f_array), color='#d62728', linewidth=2, label='Anomalía Verdadera (f)')

# Marcar la fecha del máximo acercamiento
plt.axvline(t_enc_dias, color='gray', linestyle=':', linewidth=1.5, label=f'Máximo acercamiento ({t_enc_dias:.1f} días)')

plt.xlabel('Tiempo desde el perihelio (días)', fontsize=12)
plt.ylabel('Ángulo (°)', fontsize=12)
plt.title(f'Comparación de las Tres Anomalías de Apophis (e = {e:.4f})', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## Análisis

La gráfica muestra cómo evolucionan las tres anomalías a lo largo de un período orbital completo de Apophis. La Anomalía Media ($M$) crece de forma perfectamente lineal, ya que siempre va barriendo ángulos a tasa constante $n = \sqrt{\mu/a^3}$. La Anomalía Excéntrica ($E$) y la Anomalía Verdadera ($f$), en cambio, crecen de manera no uniforme debido a la excentricidad de la órbita ($e \approx 0.19$).

Cerca del perihelio ($f \approx 0°$) la anomalía verdadera crece más rápido que $M$, reflejando que el cuerpo se desplaza con mayor velocidad cuando está más cerca del Sol, en acuerdo con la segunda ley de Kepler (ley de áreas). Cerca del afelio ($f \approx 180°$) ocurre lo contrario: $f$ crece más lento que $M$, indicando que el cuerpo se frena al alejarse del foco atractor.

## **Apophis en el CR3BP**

Debido a que la órbita de Apophis está totalmente gobernada por el sol, pero tendrá un empujón considerable debido a la tierra, es menester analizar el sistema de estos 3 cuerpos en los momentos de acercamiento de este asteroide a la tierra. Para eso, se supondrá que tanto la tierra como el sol están en un solo eje (el eje $x$) y se orbitan entre sí en órbitas casi circulares, haciendo así que el cuerpo de interés sea Apophis, ya que la órbita de la tierra respecto al sol en este caso está bien estudiada en el problema de los dos cuerpos.

Se toma que el cuerpo 1 (en este caso, el sol) se encuentra a una distancia del origen $x_1 = - \alpha$ y el cuerpo 2 (la tierra), a una distancia $x_2 = 1- \alpha$ con $\alpha = m_2 / (m_1 + m_2)$

Como los cuerpos que están en el eje $x$ se orbitan entre sí, para que siempre permanezcan sobre el eje $x$ se introduce un sistema que gira conforme ellos lo hacen. La transformación a este sistema rotante es: 
$$
x' = x\cos(\omega t) + y\sin(\omega t)
$$

$$
y' = -x\sin(\omega t) + y\cos(\omega t)
$$

con  $(x',y')$ las coordenadas en el sistema rotante,
$\omega$ es la velocidad angular con que giran los ejes, que en este caso es la velocidad orbital de la tierra.

Las ecuaciones de movimiento que surgen de esta rotación son (recordemos que al rotar el sistema deja de ser inercial, por lo que aparecen fuerzas ficticias):

$$\ddot{x} - 2\dot{y} = x - \frac{1-\alpha}{r_1^3}(x - x_1) - \frac{\alpha}{r_2^3}(x - x_2)$$

$$\ddot{y} + 2\dot{x} = y - \frac{1-\alpha}{r_1^3}\,y - \frac{\alpha}{r_2^3}$$

$$\ddot{z} = -\frac{1-\alpha}{r_1^3}\,z - \frac{\alpha}{r_2^3}$$

donde $r_1$ y $r_2$ son las distancias del cuerpo cuya trayectoria es desconocida (Apophis en este caso) hacia los otros dos cuerpos


Vamos a estudiar el movimiento de Apophis en el sistema rotante y compararlo con el movimiento en el sistema heliocéntrico, para eso debemos crear otra simulación que sólo contenga los tres cuerpos involucrados y proceder a integrar el sistema como ya se hizo anteriormente

In [ ]:
#Simulación de 3 cuerpos: Sol, Tierra y Apophis
sim_3b = rb.Simulation()
sim_3b.units = ('s', 'AU', 'Msun')
sim_3b.integrator = 'ias15'

# Fecha de inicio idéntica a la simulación de N-cuerpos
fecha_inicial = '2027-01-01 00:00:00'
tiempo_total = 4 * 365.25 * 24 * 3600  # 4 años en segundos
dt = 1 * 3600  # Paso de tiempo de 1 hora en segundos
pasos = int(tiempo_total / dt)

# Agregar únicamente Sol, Tierra (399) y Apophis
cuerpos_3b = ['Sun', '399', 'Apophis']
for c in cuerpos_3b:
    sim_3b.add(c, date=fecha_inicial)
sim_3b.move_to_com()

# Extraer masas de la simulación
m_sun = sim_3b.particles[0].m
m_earth = sim_3b.particles[1].m
m_apo = sim_3b.particles[2].m

# Reservar memoria para almacenar las trayectorias
tiempos_3b = np.zeros(pasos + 1)
posiciones_3b = np.zeros((pasos + 1, 3, 3))  # [paso, cuerpo, coordenada]
velocidades_3b = np.zeros((pasos + 1, 3, 3))

# Integrar el sistema
for paso, t in enumerate(np.linspace(0, tiempo_total, pasos + 1)):
    sim_3b.integrate(t)
    tiempos_3b[paso] = t
    for i in range(3):
        p = sim_3b.particles[i]
        posiciones_3b[paso, i] = [p.x, p.y, p.z]
        velocidades_3b[paso, i] = [p.vx, p.vy, p.vz]

# 2. Distancia mínima y fecha del encuentro en la simulación de 3 cuerpos
r_earth = posiciones_3b[:, 1, :]
r_apo = posiciones_3b[:, 2, :]
distancias_3b = np.linalg.norm(r_apo - r_earth, axis=1)
idx_min_3b = np.argmin(distancias_3b)
min_dist_km_3b = distancias_3b[idx_min_3b] * UL / 1000
fecha_min_3b = Time(fecha_inicial) + TimeDelta(tiempos_3b[idx_min_3b], format='sec')

print(f"Distancia mínima en 3 cuerpos: {min_dist_km_3b:.2f} km")
print(f"Fecha de mínimo acercamiento: {fecha_min_3b.iso}")

Primero transformamos las coordenadas a las heliocéntricas simplemente restando las posiciones de Apophis y de la tierra al sol. Luego para crear los ejes rotantes se debe crear un vector que siempre apunte del sol a la tierra, para luego definir el vector unitario como el que va en esa dirección. Se decide poner el origen de los ejes en el baricentro del sistema tierra-sol. Por último se hace la proyección de los vectores sobre el plano de rotación y se normalizan para adimensionalizar el problema.

In [ ]:
# Transformación a coordenadas Heliocéntricas (relativas al Sol)
r_apo_helio = posiciones_3b[:, 2, :] - posiciones_3b[:, 0, :]
r_earth_helio = posiciones_3b[:, 1, :] - posiciones_3b[:, 0, :]

# ransformación al Sistema Rotante
r_sun = posiciones_3b[:, 0, :]
r_se = r_earth - r_sun  # Vector Sol - Tierra
v_se = velocidades_3b[:, 1, :] - velocidades_3b[:, 0, :]  # Velocidad relativa Tierra-Sol

# Ejes rotantes instantáneos
x_rot = r_se / np.linalg.norm(r_se, axis=1)[:, np.newaxis]
h_vec = np.cross(r_se, v_se)
z_rot = h_vec / np.linalg.norm(h_vec, axis=1)[:, np.newaxis]
y_rot = np.cross(z_rot, x_rot)

# Origen en el baricentro del sistema Tierra-Sol
r_bar = (m_sun * r_sun + m_earth * r_earth) / (m_sun + m_earth)
r_apo_bar = r_apo - r_bar

# Proyección sobre el plano de rotación (x_syn, y_syn)
x_syn = np.sum(r_apo_bar * x_rot, axis=1)
y_syn = np.sum(r_apo_bar * y_rot, axis=1)
z_syn = np.sum(r_apo_bar * z_rot, axis=1)  

# Normalizar por la distancia instantánea Tierra-Sol R(t) para adimensionalizar
R = np.linalg.norm(r_se, axis=1)
x_syn_nd = x_syn / R
y_syn_nd = y_syn / R
z_syn_nd = z_syn / R
# ===================== GRAFICACIÓN =====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), dpi=100)

# Colores y estilo
c_sun = '#f39c12'
c_earth = '#2980b9'
c_apo = '#e74c3c'
c_apo_line = '#2c3e50'
c_grid = '#bdc3c7'

# Subplot 1: Sistema Heliocéntrico (Inercial)
ax1.plot(r_earth_helio[:, 0], r_earth_helio[:, 1], color=c_earth, linestyle='--', alpha=0.8, label='Órbita de la Tierra')
ax1.plot(r_apo_helio[:, 0], r_apo_helio[:, 1], color=c_apo_line, linewidth=1.5, label='Trayectoria de Apophis')
ax1.scatter([0], [0], color=c_sun, s=150, zorder=5, label='Sol', edgecolors='black')
ax1.scatter(r_earth_helio[idx_min_3b, 0], r_earth_helio[idx_min_3b, 1], color=c_earth, s=80, zorder=5, label='Tierra (Encuentro)', edgecolors='black')
ax1.scatter(r_apo_helio[idx_min_3b, 0], r_apo_helio[idx_min_3b, 1], color=c_apo, s=80, zorder=5, label='Apophis (Encuentro)', edgecolors='black')

ax1.set_xlabel('X (AU)', fontsize=11)
ax1.set_ylabel('Y (AU)', fontsize=11)
ax1.set_title('Sistema Heliocéntrico (Inercial)', fontsize=13, fontweight='bold', pad=10)
ax1.grid(True, linestyle=':', color=c_grid, alpha=0.6)
ax1.legend(loc='upper right', fontsize=9)
ax1.set_aspect('equal')

# Subplot 2: Sistema Rotante Tierra-Sol (Sinódico)
alpha = m_earth / (m_sun + m_earth)
x_sun_rot = -alpha
x_earth_rot = 1.0 - alpha

ax2.plot(x_syn_nd, y_syn_nd, color=c_apo_line, linewidth=1.5, label='Trayectoria de Apophis')
ax2.scatter([x_sun_rot], [0], color=c_sun, s=150, zorder=5, label='Sol', edgecolors='black')
ax2.scatter([x_earth_rot], [0], color=c_earth, s=80, zorder=5, label='Tierra', edgecolors='black')
ax2.scatter(x_syn_nd[idx_min_3b], y_syn_nd[idx_min_3b], color=c_apo, s=80, zorder=6, label='Apophis (Encuentro)', edgecolors='black')

ax2.set_xlabel("X' (Unidades de distancia $R_{TS}$)", fontsize=11)
ax2.set_ylabel("Y' (Unidades de distancia $R_{TS}$)", fontsize=11)
ax2.set_title('Sistema Rotante Tierra-Sol (Sinódico Normalizado)', fontsize=13, fontweight='bold', pad=10)
ax2.grid(True, linestyle=':', color=c_grid, alpha=0.6)
ax2.legend(loc='upper right', fontsize=9)
ax2.set_aspect('equal')

plt.suptitle('Comparación del Movimiento de Apophis en 3 Cuerpos (Sol - Tierra - Apophis)', fontsize=15, fontweight='bold', y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


### Análisis
Al comparar la trayectoria de Apophis en ambos sistemas de referencia, podemos ver que:
- Todas las órbitas de Apophis respecto al sol con todos los métodos que se ha realizado han sido consecuentes, lo cual confirma su validez.
-  Al fijar la línea Sol-Tierra y rotar el sistema de coordenadas a la velocidad angular orbital de la Tierra: Apophis, al tener un período orbital menor al de la Tierra ($T \approx 0.88$ años), orbita más rápido. En el plano rotante, esto se ve en la forma espiral de la trayectoria.
- En esta simulación simplificada de sólo 3 cuerpos, la distancia mínima de acercamiento resulta ser de aproximadamente **$651,694\text{ km}$** en abril de 2029. Esto contrasta fuertemente con los **$38,346\text{ km}$** obtenidos en la simulación de N-cuerpos. Esta diferencia radical evidencia que las perturbaciones acumuladas a lo largo de los dos años previos por parte de los demás planetas y la presencia de la Luna son fundamentales para guiar la trayectoria precisa del asteroide. Sin estas perturbaciones secundarias, el asteroide y la Tierra no llegan a cruzarse a una distancia tan extrema en abril de 2029.


## **Condición de Equilibrio**
En el sistema de referencia rotante, la condición para que una partícula permanezca en reposo relativo (es decir, en equilibrio, con $\dot{x}=\dot{y}=\dot{z}=0$ y $\ddot{x}=\ddot{y}=\ddot{z}=0$) requiere que las fuerzas gravitatorias de los dos cuerpos primarios y la fuerza centrífuga se cancelen mutuamente. Matemáticamente, esto equivale a buscar los puntos críticos o estacionarios de la superficie de potencial modificado $V_{mod}$:
$$\nabla V_{mod} = 0 \quad \Longrightarrow \quad \frac{\partial V_{mod}}{\partial x} = 0, \quad \frac{\partial V_{mod}}{\partial y} = 0, \quad \frac{\partial V_{mod}}{\partial z} = 0$$

con $V_{mod}$ el potencial modificado que tiene la forma:
$$V_{mod} = - \frac{1 - \alpha}{r_1} - \frac{\alpha}{r_2} - \frac{x^2 + y^2}{2}$$


Al resolver estas ecuaciones en tres dimensiones, se obtienen exactamente cinco puntos de equilibrio, conocidos como los Puntos de Lagrange ($L_1$ a $L_5$):

Puntos Colineales ($L_1, L_2, L_3$): Son aquellos que yacen sobre la línea de unión de los dos primarios (el eje $x$, con $y=0$, $z=0$).

$L_1$: Se ubica en la región intermedia entre ambos cuerpos primarios.
$L_2$: Se encuentra en el lado exterior del cuerpo secundario (la Tierra en este caso).
$L_3$: Se sitúa en el lado opuesto del cuerpo primario principal (el Sol).
Dinámicamente, estos tres puntos son inestables y actúan como puntos de silla (saddle points) en la superficie de potencial, lo que significa que cualquier perturbación menor hará que un objeto abandone la zona de equilibrio.

Puntos Triangulares ($L_4, L_5$): Son aquellos que forman un triángulo equilátero con los dos primarios en el plano orbital ($z=0$), situándose a $60^\circ$ delante ($L_4$) y $60^\circ$ detrás ($L_5$) del cuerpo secundario en su órbita.

La ecuación que define a los puntos colineales son: 

$$ x - \frac{(1-\alpha)(x+\alpha)}{|x+\alpha|^3} - \frac{\alpha(x-1+\alpha)}{|x-1+\alpha|^3} = 0$$

La cual no tiene solución analítica, en el código se implementó el método de bisección para resolver numéricamente esta ecuación

Por otro lado, los puntos triangulares se hallan en los siguientes puntos:
$$L_4 = \left(\tfrac{1}{2} - \alpha,\; +\frac{\sqrt{3}}{2},\; 0\right), \qquad L_5 = \left(\tfrac{1}{2} - \alpha,\; -\frac{\sqrt{3}}{2},\; 0\right)$$

Para graficar estos puntos se hace una gráfica decontornos donde también se señalan los puntos de Lagrange

In [ ]:
## Graficos del potencial modificado
alfa = m_earth / (m_sun + m_earth)  # Parámetro de masa (alfa)
x1 = -alfa # Posición del Sol en el sistema rotante
x2 = 1 - alfa # Posición de la Tierra en el sistema rotante

Ng = 100
vmax = 1.5
xs = np.linspace(-vmax, vmax, Ng)
ys = np.linspace(-vmax, vmax, Ng)

Xs, Ys = np.meshgrid(xs, ys)
Vmod = np.zeros((Ng, Ng))
soft = 0.1

for iy in range(Ng):
  for ix in range(Ng):
    x = Xs[iy, ix]
    y = Ys[iy, ix]
    
    r1 = np.linalg.norm([x-x1, y]) + soft
    r2 = np.linalg.norm([x-x2, y]) + soft
    vmod = -(1-alfa)/r1 - alfa/r2 - 0.5*(x**2 + y**2)
    Vmod[iy, ix] = vmod

# ===================== GRAFICACIÓN =====================
fig, ax1 = plt.subplots(1, 1, figsize=(16, 7.5), dpi=100)

# 1. Superficie de Potencial Modificado Vmod (Contornos)
# Graficamos contornos limitados para observar la estructura de los pozos y puntos de Lagrange
Vmod_capped = np.clip(Vmod, -2.0, -1.4)
contour = ax1.contourf(Xs, Ys, Vmod_capped, levels=30, cmap='viridis', alpha=0.85)
cbar = fig.colorbar(contour, ax=ax1)
cbar.set_label('Potencial Modificado $V_{mod}$ (adimensional)', fontsize=11)

# Trayectoria de Apophis
ax1.plot(x_syn_nd, y_syn_nd, color='#e74c3c', linewidth=1.5, label='Trayectoria de Apophis', zorder=4)

# Sol y Tierra
ax1.scatter([x1], [0], color='#f39c12', s=180, zorder=5, label='Sol', edgecolors='black')
# Calcular puntos de Lagrange exactos para mostrar en el gráfico
def eq_collinear(x):
    term1 = (1 - alfa) * (x - x1) / (np.abs(x - x1)**3)
    term2 = alfa * (x - x2) / (np.abs(x - x2)**3)
    return x - term1 - term2

def bisection(func, a, b, tol=1e-12, max_iter=100):
    fa, fb = func(a), func(b)
    if fa * fb > 0: return None
    for _ in range(max_iter):
        c = (a + b) / 2.0
        fc = func(c)
        if np.abs(fc) < tol or (b - a)/2.0 < tol: return c
        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc
    return (a + b) / 2.0

L1 = bisection(eq_collinear, x1 + 1e-6, x2 - 1e-6)
L2 = bisection(eq_collinear, x2 + 1e-6, 2.0)
L3 = bisection(eq_collinear, -2.0, x1 - 1e-6)
L4_x, L4_y = 0.5 - alfa, np.sqrt(3)/2.0
L5_x, L5_y = 0.5 - alfa, -np.sqrt(3)/2.0

ax1.scatter([L1, L2, L3], [0, 0, 0], color='white', s=60, marker='x', zorder=6, label='Puntos Colineales $L_1, L_2, L_3$')
ax1.scatter([L4_x, L5_x], [L4_y, L5_y], color='white', s=60, marker='d', zorder=6, label='Puntos Triangulares $L_4, L_5$')

ax1.set_xlabel("X' (Unidades de distancia $R_{TS}$)", fontsize=11)
ax1.set_ylabel("Y' (Unidades de distancia $R_{TS}$)", fontsize=11)
ax1.set_title("Superficie del Potencial Modificado $V_{mod}$", fontsize=13, fontweight='bold', pad=10)
ax1.legend(loc='lower left', fontsize=9)
ax1.set_aspect('equal')
ax1.set_xlim(-vmax, vmax)
ax1.set_ylim(-vmax, vmax)

fechas_time = Time(fecha_inicial) + TimeDelta(tiempos_3b, format='sec')

plt.suptitle('Potencial Modificado en el Sistema Sol-Tierra-Apophis', fontsize=15, fontweight='bold', y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


## Análisis
El mapa de contornos ilustra el relieve energético en el que se desplaza el asteroide. El Sol y la Tierra ocupan los centros de atracción correspondientes a los dos profundos pozos de potencial (regiones de menor valor energético).
En este mapa se destacan los Puntos de Lagrange ($L_1, L_2, L_3, L_4, L_5$), que son los puntos de equilibrio donde el gradiente de la superficie de potencial modificado es nulo ($\nabla V_{mod} = 0$). En ellos, la fuerza centrífuga compensa exactamente la atracción gravitatoria combinada de los primarios.
La órbita de Apophis en el marco rotante avanza rodeando al Sol, pasando de forma dinámica a través de los contornos energéticos y esquivando los puntos de equilibrio.

## **Conclusiones**

- Se observó para el caso del problema de N cuerpos que, aún sin incluir la totalidad de la masa del sistema solar, se pudo predecir con bastante precisión la distancia y la fecha del paso de Apophis por la tierra, lo que también permite afirmar que, al menos para el caso de Apophis, el sol, la luna, la tierra y los demás planetas son los que controlan casi en su totalidad al asteroide.
- Se confirmó que, aunque en el sistema solar hayan encuentros casi colisionales entre objetos, el momentum angular permanece constante siempre, cosa que no pasa con el momentum angular específico entre Apophis y el sol, que tuvo un salto en el momento que la tierra lo afectó gravitacionalmente. 

- La energía potencial y cinética se contrarrestan casi a la perfección para hacer de la energía mecánica una constante.

- Por el lado del problema de los dos cuerpos, vimos que la órbita de Apophis respecto al sol que se obtuvo a partir de la teoría de los dos cuerpos fue bastante acertada al compararla con las efemérides.
- Se observó el comportamiento de las tres anomalías, verificando que la anomalía media es, como su nombre lo dice, la que evoluciona de manera lineal, mientras que las otras dos crecen o disminuyen dependiendo de la posición en que se encuentre el cuerpo respecto al sol, en este caso.

- Por último, se notó que, usando la teoría del problema de los 3 cuerpos, la distancia mínima que registra Apophis respecto a la tierra tiene un desfase muy grande respecto a la real, esto debido a que sólo se están tomando en cuenta dos cuerpos en la trayectoria de Apophis, que si bien son muy importantes, no dominan completamente la trayectoria de éste.

## **Referencias**

1.   Zuluaga, Jorge I. (2024). MECANICA CELESTE; TEORIA, ALGORITMOS Y PROBLEMAS. UNIVERSIDAD DE ANTIOQUIA.

2. JPL HORIZONS On-Line Solar System Data and Ephemeris Computation Service. NASA/JPL, Pasadena, California. URL. https://ssd.jpl.nasa.gov/.
